# Lab 9 and Lab 10
**Name:** Avni Singhal &nbsp;&nbsp; **Roll number:** 202451026


## Lab 9 — Question 1: MENACE (Tic-Tac-Toe)

**Question:** Read Donald Michie's MENACE paper, examine an existing implementation and identify its important code. If possible, implement MENACE.

**What MENACE does:** It learns to play Tic-Tac-Toe by remembering which moves worked. A board position acts as a *matchbox*, and a move's *bead count* determines how likely that move is to be chosen. After each game, beads for chosen moves change according to the outcome.


In [2]:
"""AI Lab 9 - Q1: Simple MENACE (matchbox tic-tac-toe learning)"""

import random
WIN_LINES = (
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
)

def result(board):
    """Return X, O, DRAW, or None (game not yet over)"""
    for a, b, c in WIN_LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    if " " not in board:
        return "DRAW"
    return None

class Menace:
    def __init__(self, seed=10):
        self.rng = random.Random(seed)
        self.matchboxes = {}  # board string -> {available cell: number of beads}

    def choose(self, board):
        state = "".join(board)
        if state not in self.matchboxes:
            self.matchboxes[state] = {
                index: 3 for index, cell in enumerate(board) if cell == " "
            }
        beads = self.matchboxes[state]
        moves = list(beads)
        weights = [beads[move] for move in moves]
        # Always allow the available moves, even if every bead was removed.
        if sum(weights) == 0:
            weights = [1] * len(moves)
        move = self.rng.choices(moves, weights=weights, k=1)[0]
        return state, move

    def learn(self, history, outcome):
        """Reward a win +3 beads, draw +1, loss -1 (minimum 0)"""
        change = {"X": 3, "DRAW": 1, "O": -1}[outcome]
        for state, move in history:
            beads = self.matchboxes[state]
            beads[move] = max(0, beads[move] + change)

def one_game(agent, opponent_rng, training):
    board = [" "] * 9
    history = []
    while result(board) is None:
        state, move = agent.choose(board)
        board[move] = "X"  # MENACE plays first
        history.append((state, move))
        if result(board) is not None:
            break
        available = [i for i, cell in enumerate(board) if cell == " "]
        board[opponent_rng.choice(available)] = "O"  # random opponent
    outcome = result(board)
    if training:
        agent.learn(history, outcome)
    return outcome

def experiment(agent, games, training, seed):
    opponent_rng = random.Random(seed)
    scores = {"X": 0, "O": 0, "DRAW": 0}
    for _ in range(games):
        scores[one_game(agent, opponent_rng, training)] += 1
    return scores

def show_scores(title, scores, total):
    print(title)
    print(f"  Wins  : {scores['X']} ({scores['X'] / total:.1%})")
    print(f"  Draws : {scores['DRAW']} ({scores['DRAW'] / total:.1%})")
    print(f"  Losses: {scores['O']} ({scores['O'] / total:.1%})")

def main():
    agent = Menace(seed=21)
    test_games = 2000
    training_games = 30000
    before = experiment(agent, test_games, training=False, seed=100)
    experiment(agent, training_games, training=True, seed=101)
    after = experiment(agent, test_games, training=False, seed=102)
    print("LAB 9 - Q1 | MENACE versus a random opponent")
    show_scores("Before training:", before, test_games)
    print(f"Trained for {training_games} games.")
    show_scores("After training:", after, test_games)
    print(f"Learned matchboxes: {len(agent.matchboxes)}")

main()

LAB 9 - Q1 | MENACE versus a random opponent
Before training:
  Wins  : 1149 (57.5%)
  Draws : 267 (13.4%)
  Losses: 584 (29.2%)
Trained for 30000 games.
After training:
  Wins  : 1687 (84.4%)
  Draws : 164 (8.2%)
  Losses: 149 (7.4%)
Learned matchboxes: 2412


**What we did / result:** Trained MENACE by playing against a random opponent, then compared the win, draw and loss rates before and after training. A higher win rate after training demonstrates learning through reinforcement.

## Lab 9 — Question 2: Binary bandit using epsilon-greedy

**Aim:** Select action **1** or **2** in a binary stochastic bandit and learn which gives a higher expected reward. Every pull returns **1 = success** or **0 = failure**, with fixed (stationary) reward probabilities.


**Method:** With probability `epsilon = 0.1`, select a random action (**exploration**). Otherwise select a current best estimated action (**exploitation**), breaking ties randomly. Update the chosen action's estimate after every pull:

\[
Q_{new}(a)=Q_{old}(a)+\frac{R-Q_{old}(a)}{N(a)}
\]

Here `Q(a)` is the estimated success rate and `N(a)` is the number of times action `a` has been selected. Run epsilon-greedy for both environments independently and print the learned best action.


In [3]:
import random

class BinaryBanditExample:
    """Two actions (1, 2); each returns success (1) or failure (0)"""
    def __init__(self, p_action1, p_action2, seed):
        self.probabilities = (p_action1, p_action2)
        self.rng = random.Random(seed)

    def pull(self, action):
        if action not in (1, 2):
            raise ValueError('Action must be 1 or 2')
        return int(self.rng.random() < self.probabilities[action - 1])

# Two independent, stationary test environments
example_A = BinaryBanditExample(0.30, 0.70, seed=101)
example_B = BinaryBanditExample(0.80, 0.40, seed=102)

def binaryBanditA(action):
    return example_A.pull(action)

def binaryBanditB(action):
    return example_B.pull(action)

def epsilon_greedy_binary(pull, steps=10_000, epsilon=0.1, seed=7):
    """Learn the better of two actions using only sampled binary rewards."""
    rng = random.Random(seed)
    Q = [0.0, 0.0]       # Initial estimates for actions 1 and 2
    N = [0, 0]           # Number of pulls of each action
    successes = 0

    for _ in range(steps):
        if rng.random() < epsilon:
            index = rng.randrange(2)  # Explore
        else:
            highest = max(Q)
            tied = [i for i in range(2) if Q[i] == highest]
            index = rng.choice(tied)  # Exploit, random tie-break

        action = index + 1
        reward = pull(action)  # A reward is only 0 or 1
        if reward not in (0, 1):
            raise ValueError('Binary bandit must return 0 or 1')

        successes += reward
        N[index] += 1
        Q[index] += (reward - Q[index]) / N[index]  # Sample average

    return {
        'estimates': Q,
        'counts': N,
        'successes': successes,
        'steps': steps,
        'best_action': Q.index(max(Q)) + 1,
    }

for name, bandit, known_best in [
    ('A', binaryBanditA, 2),
    ('B', binaryBanditB, 1),
]:
    result = epsilon_greedy_binary(bandit, steps=10_000, seed=7)
    print(f'\nBinary Bandit {name} ')
    print(f"  Estimated Q(action 1) = {result['estimates'][0]:.4f}")
    print(f"  Estimated Q(action 2) = {result['estimates'][1]:.4f}")
    print(f"  Action counts = {result['counts']}")
    print(f"  Total successes = {result['successes']} / {result['steps']}")
    print(f"  Learned best action = {result['best_action']}")
    print(f"  Known better action = {known_best}")
    assert result['best_action'] == known_best, 'Expected best action not learned'


Binary Bandit A 
  Estimated Q(action 1) = 0.2838
  Estimated Q(action 2) = 0.6901
  Action counts = [525, 9475]
  Total successes = 6688 / 10000
  Learned best action = 2
  Known better action = 2

Binary Bandit B 
  Estimated Q(action 1) = 0.8071
  Estimated Q(action 2) = 0.3644
  Action counts = [9506, 494]
  Total successes = 7852 / 10000
  Learned best action = 1
  Known better action = 1


**What we did / result:** Created two stationary two-action binary bandit environments and applied the **same epsilon-greedy learner independently** to each. Every reward was either 0 or 1. The printed estimates show that the agent learned to prefer **action 2 in example A** and **action 1 in example B**. Exploration helped the agent discover the better action; exploitation helped it obtain more successful pulls.


## Question 3: Create a non-stationary 10-armed bandit

**Question:** Initially all **10 true mean rewards are equal**. At every time step, each mean independently changes by a Gaussian random increment with **mean 0** and **standard deviation 0.01**. The function receives an action from **1 to 10** and returns its reward.


In [4]:
import random

class NonStationaryBandit:
    def __init__(self, seed=100):
        self.rng = random.Random(seed)
        self.true_means = [0.0] * 10  # All 10 start equally

    def bandit_nonstat(self, action):
        """Choose action 1..10, receive reward, then drift every arm's true mean"""
        if action not in range(1, 11):
            raise ValueError('action must be 1 through 10')

        reward = self.rng.gauss(self.true_means[action - 1], 1.0)
        for i in range(10):
            self.true_means[i] += self.rng.gauss(0.0, 0.01)
        return reward

nonstat_env = NonStationaryBandit(seed=99)
def bandit_nonstat(action):
    return nonstat_env.bandit_nonstat(action)

print('Initial true means:', [round(x, 4) for x in nonstat_env.true_means])
for action in [1, 5, 10, 2, 7]:
    print(f'action={action:2d}, observed reward={bandit_nonstat(action):+.4f}')
print('True means after five steps:', [round(x, 4) for x in nonstat_env.true_means])

Initial true means: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
action= 1, observed reward=-0.5502
action= 5, observed reward=+0.0629
action=10, observed reward=+1.1411
action= 2, observed reward=+0.5071
action= 7, observed reward=-0.1244
True means after five steps: [-0.0295, -0.0052, 0.0009, -0.0044, -0.0131, -0.0013, 0.0244, -0.0119, 0.0237, -0.0471]


**What we did / result:** Created 10 actions with equal initial expected rewards. Each call returns a stochastic reward and moves **all ten** means by separate small random steps, so the best action can change over time. The output above verifies that the means are no longer equal.

## Question 4: Modified epsilon-greedy for the changing bandit

**Question:** Improve epsilon-greedy so that it can track the best action when the rewards change. **Test at least 10,000 time steps** and say whether it follows the correct actions.

A normal agent averages all old rewards: `α = 1/N(a)`. A modified agent uses a **constant step size `α = 0.1`**, keeping recent experience important:

\[
Q_{t+1}(a) = Q_t(a) + 0.1\,(R_t-Q_t(a))
\]

We compare the two agents on identical-seed copies of the non-stationary bandit from Question 3. Both use `ε = 0.1`. We measure how often the agent chooses the **currently highest true-mean action** in the last 1,000 of 10,000 steps.

In [5]:
def choose_epsilon_greedy(Q, rng, epsilon=0.1):
    if rng.random() < epsilon:
        return rng.randrange(10)
    best = max(Q)
    tied = [i for i, value in enumerate(Q) if value == best]
    return rng.choice(tied)

def test_nonstationary_agent(seed, constant_alpha, steps=10_000):
    environment = NonStationaryBandit(seed=seed)
    agent_rng = random.Random(seed + 1234)
    Q = [0.0] * 10
    counts = [0] * 10
    correct_last_1000 = 0
    reward_total = 0.0

    for t in range(steps):
        # Identify the best action BEFORE the true means drift on this step.
        current_best_mean = max(environment.true_means)
        best_actions = {i for i, mean in enumerate(environment.true_means)
                        if mean == current_best_mean}
        action_index = choose_epsilon_greedy(Q, agent_rng)
        reward = environment.bandit_nonstat(action_index + 1)

        counts[action_index] += 1
        alpha = 0.1 if constant_alpha else 1.0 / counts[action_index]
        Q[action_index] += alpha * (reward - Q[action_index])
        reward_total += reward

        if t >= steps - 1000 and action_index in best_actions:
            correct_last_1000 += 1

    return correct_last_1000 / 1000, reward_total / steps

STEPS = 10_000
RUNS = 10       # Repeat to make the conclusion less dependent on one random run
for label, modified in [('Normal sample-average', False),
                        ('Modified constant-alpha', True)]:
    results = [test_nonstationary_agent(400 + i, modified, STEPS)
               for i in range(RUNS)]
    correct = sum(x[0] for x in results) / RUNS
    avg_reward = sum(x[1] for x in results) / RUNS
    print(f'{label:24s} | optimal action (last 1000): {correct:.1%} '
          f'| average reward: {avg_reward:.3f}')

Normal sample-average    | optimal action (last 1000): 19.1% | average reward: 0.758
Modified constant-alpha  | optimal action (last 1000): 73.6% | average reward: 0.951


**What we did / result:** We tested 10,000 steps in each experiment and compared ordinary and modified learning. The modified agent usually tracks changing action values more effectively, although it still sometimes explores or selects a non-optimal action.

# Lab 10 — Markov Decision Process and Dynamic Programming

## Question 1: 4×3 Gridworld using value iteration

**Question:** Use the specified 4×3 map (start at bottom left, wall in the middle, terminals +1 and −1), moves Up/Down/Left/Right, probabilities **0.8 intended**, **0.1 each perpendicular direction**, and remain in place if a wall/boundary is hit. Determine the optimal **value function** for **five non-terminal rewards**: `−0.04`, `−2`, `0.1`, `0.02`, `1`.

We repeatedly apply the Bellman optimality update and display both the state values and corresponding preferred actions for each reward.

**Assumption required to calculate numbers:** The PDF does **not** specify a discount factor for Gridworld. We use **γ = 0.9** for these five calculations and state it explicitly. This is an assumption. Terminal rewards are received **on entering** the terminal cells, as described in the handout.

In [6]:
"""AI Lab 10 - Q1: Value iteration in the 4x3 stochastic Gridworld"""

ACTIONS = {
    "UP": (0, 1),
    "RIGHT": (1, 0),
    "DOWN": (0, -1),
    "LEFT": (-1, 0),
}
SIDEWAYS = {
    "UP": ("LEFT", "RIGHT"),
    "RIGHT": ("UP", "DOWN"),
    "DOWN": ("LEFT", "RIGHT"),
    "LEFT": ("UP", "DOWN"),
}
ARROWS = {"UP": "^", "RIGHT": ">", "DOWN": "v", "LEFT": "<"}
WALL = (1, 1)
TERMINALS = {(3, 2): +1.0, (3, 1): -1.0}
STATES = [(x, y) for x in range(4) for y in range(3)
          if (x, y) != WALL]

def move(state, action):
    dx, dy = ACTIONS[action]
    next_state = (state[0] + dx, state[1] + dy)
    if not (0 <= next_state[0] < 4 and 0 <= next_state[1] < 3):
        return state
    if next_state == WALL:
        return state
    return next_state

def expected_value(state, action, values, living_reward, gamma):
    """Bellman expectation for one state and one attempted action"""
    choices = [(action, 0.8)] + [(side, 0.1) for side in SIDEWAYS[action]]
    total = 0.0
    for actual_action, probability in choices:
        next_state = move(state, actual_action)
        if next_state in TERMINALS:
            reward = TERMINALS[next_state]
            future = 0.0  # terminal state has no future rewards
        else:
            reward = living_reward
            future = gamma * values[next_state]
        total += probability * (reward + future)
    return total

def value_iteration(living_reward, gamma=0.9, tolerance=1e-11):
    values = {state: 0.0 for state in STATES}
    for iteration in range(1, 100001):
        new_values = values.copy()
        for state in STATES:
            if state in TERMINALS:
                continue  # keep the continuation value of terminals at zero
            new_values[state] = max(
                expected_value(state, a, values, living_reward, gamma)
                for a in ACTIONS
            )
        max_change = max(abs(new_values[s] - values[s]) for s in STATES)
        values = new_values
        if max_change < tolerance:
            break
    policy = {}
    for state in STATES:
        if state not in TERMINALS:
            policy[state] = max(
                ACTIONS,
                key=lambda a: expected_value(state, a, values, living_reward, gamma),
            )
    return values, policy, iteration

def show_grid(values, policy, living_reward, iterations):
    print(f"\nLiving reward r = {living_reward:5.2f} | iterations = {iterations}")
    print("Optimal value function (terminal squares display their reward):")
    for y in (2, 1, 0):
        cells = []
        for x in range(4):
            s = (x, y)
            cells.append(
                " WALL  " if s == WALL else
                f"{TERMINALS[s]:7.3f}" if s in TERMINALS else
                f"{values[s]:7.3f}"
            )
        print("  ".join(cells))
    print("Optimal action policy (^ up, > right, v down, < left):")
    for y in (2, 1, 0):
        row = []
        for x in range(4):
            s = (x, y)
            row.append(
                "#" if s == WALL else
                "+1" if s == (3, 2) else
                "-1" if s == (3, 1) else ARROWS[policy[s]]
            )
        print("    ".join(f"{item:>2}" for item in row))
    print("Start-state value:", round(values[(0, 0)], 4))

def main():
    print("4x3 Gridworld (assumed gamma=0.9)")
    for living_reward in (-0.04, -2.0, 0.1, 0.02, 1.0):
        values, policy, iterations = value_iteration(living_reward)
        show_grid(values, policy, living_reward, iterations)

main()

4x3 Gridworld (assumed gamma=0.9)

Living reward r = -0.04 | iterations = 34
Optimal value function (terminal squares display their reward):
  0.610    0.766    0.928    1.000
  0.487   WALL      0.585   -1.000
  0.374    0.327    0.428    0.189
Optimal action policy (^ up, > right, v down, < left):
 >     >     >    +1
 ^     #     ^    -1
 ^     >     ^     <
Start-state value: 0.3739

Living reward r = -2.00 | iterations = 32
Optimal value function (terminal squares display their reward):
 -4.563   -2.184    0.291    1.000
 -6.446   WALL     -1.505   -1.000
 -7.320   -5.668   -3.678   -1.682
Optimal action policy (^ up, > right, v down, < left):
 >     >     >    +1
 ^     #     >    -1
 >     >     >     ^
Start-state value: -7.3201

Living reward r =  0.10 | iterations = 119
Optimal value function (terminal squares display their reward):
  1.000    1.000    1.000    1.000
  1.000   WALL      1.000   -1.000
  1.000    1.000    1.000    1.000
Optimal action policy (^ up, > right, v 

**What we did / result:** For each of the five requested rewards, the output shows the entire 4×3 value grid and the direction preferred in each non-terminal state. When living rewards become positive, the agent may prefer remaining in non-terminal states instead of exiting. The values apply to the stated `γ = 0.9` assumption.

## Lab 10 — Question 2: Formulate the Gbike rental MDP

**Question:** Formulate a **continuing finite MDP** with daily time steps, states counting bikes at two locations, overnight transfer actions, Poisson rental requests and returns, and reward based on income minus moving cost.

**Other given data:** ₹10 per successful rental; ₹2 per bike transferred overnight; maximum **5** bikes transferred per night; discount **γ = 0.9**.


In [7]:
import math
import numpy as np

MAX_BIKES = 20
MAX_MOVE = 5
DISCOUNT = 0.9
RENT_INCOME = 10.0
MOVE_COST = 2.0
PARKING_COST = 4.0
PARKING_THRESHOLD = 10
POSITIONS = np.arange(MAX_BIKES + 1)
ACTIONS = np.arange(-MAX_MOVE, MAX_MOVE + 1)

def capped_poisson(mean):
    """P(0), ..., P(19), P(20 or more) for a Poisson random variable."""
    probs = [math.exp(-mean)]
    for k in range(1, MAX_BIKES):
        probs.append(probs[-1] * mean / k)
    probs.append(max(0.0, 1.0 - sum(probs)))
    return np.asarray(probs)

def location_model(demand_mean, return_mean):
    """For 0..20 available bikes return expected rent and next-state probabilities."""
    request_probs = capped_poisson(demand_mean)
    return_probs = capped_poisson(return_mean)
    expected_rent = np.zeros(MAX_BIKES + 1)
    transitions = np.zeros((MAX_BIKES + 1, MAX_BIKES + 1))

    for available in range(MAX_BIKES + 1):
        for requests in range(MAX_BIKES + 1):
            rented = min(available, requests)
            expected_rent[available] += request_probs[requests] * rented
            left = available - rented
            for returned in range(MAX_BIKES + 1):
                end = min(MAX_BIKES, left + returned)
                transitions[available, end] += (
                    request_probs[requests] * return_probs[returned]
                )
    assert np.allclose(transitions.sum(axis=1), 1.0, atol=1e-10)
    return expected_rent * RENT_INCOME, transitions

class GbikeMDP:
    def __init__(self, modified):
        self.modified = modified
        self.reward1, self.prob1 = location_model(3, 3)
        self.reward2, self.prob2 = location_model(4, 2)
        i, j = np.meshgrid(POSITIONS, POSITIONS, indexing="ij")
        self.after1 = np.zeros((len(ACTIONS), 21, 21), dtype=int)
        self.after2 = np.zeros_like(self.after1)
        self.reward = np.full((len(ACTIONS), 21, 21), -np.inf)
        self.valid = np.zeros((len(ACTIONS), 21, 21), dtype=bool)

        for ai, action in enumerate(ACTIONS):
            # Positive action transfers location 1 -> location 2.
            post1, post2 = i - action, j + action
            valid = ((post1 >= 0) & (post1 <= MAX_BIKES) &
                     (post2 >= 0) & (post2 <= MAX_BIKES))
            x = np.clip(post1, 0, MAX_BIKES)
            y = np.clip(post2, 0, MAX_BIKES)
            self.after1[ai] = x
            self.after2[ai] = y
            self.valid[ai] = valid

            # Standard transfer cost is 2 INR per bicycle either direction.
            cost = MOVE_COST * abs(action)
            if modified and action > 0:
                cost = MOVE_COST * max(0, action - 1)  # first 1->2 bike free
            parking = 0.0
            if modified:
                parking = PARKING_COST * ((post1 > PARKING_THRESHOLD).astype(float)
                                          + (post2 > PARKING_THRESHOLD).astype(float))
            immediate = self.reward1[x] + self.reward2[y] - cost - parking
            self.reward[ai] = np.where(valid, immediate, -np.inf)

    def all_action_values(self, values):
        """Q(s,a) = expected immediate reward + 0.9 * expected next value."""
        future = self.prob1 @ values @ self.prob2.T
        q_values = self.reward + DISCOUNT * future[self.after1, self.after2]
        return np.where(self.valid, q_values, -np.inf)

    def policy_iteration(self, tolerance=1e-8):
        values = np.zeros((21, 21))
        policy_indices = np.full((21, 21), MAX_MOVE, dtype=int)  # a=0 initially
        indices = np.indices((21, 21))
        rows, cols = indices[0], indices[1]

        for improvement in range(1, 51):
            # Policy evaluation: repeatedly apply Bellman's expectation equation.
            for evaluation in range(1, 2001):
                future = self.prob1 @ values @ self.prob2.T
                new_values = (
                    self.reward[policy_indices, rows, cols]
                    + DISCOUNT * future[
                        self.after1[policy_indices, rows, cols],
                        self.after2[policy_indices, rows, cols],
                    ]
                )
                delta = np.max(np.abs(new_values - values))
                values = new_values
                if delta < tolerance:
                    break

            # Policy improvement: for every state, choose the highest-Q action.
            new_policy = np.argmax(self.all_action_values(values), axis=0)
            changed = int(np.sum(new_policy != policy_indices))
            print(f"Policy improvement {improvement}: changed states = {changed}; "
                  f"evaluation sweeps = {evaluation}")
            policy_indices = new_policy
            if changed == 0:
                break
        else:
            raise RuntimeError("Policy iteration failed to stabilize in 50 improvements")
        return ACTIONS[policy_indices], values

print('Gbike finite MDP defined: 441 states and up to 11 candidate transfer actions per state.')

Gbike finite MDP defined: 441 states and up to 11 candidate transfer actions per state.


In [8]:
# Question 2: solve the original Gbike model.
# The textbook's Jack's Car Rental Example 4.2 uses these same parameters.
basic_gbike = GbikeMDP(modified=False)
basic_policy, basic_values = basic_gbike.policy_iteration()

print('\nOriginal Gbike: optimal overnight transfers for all 441 states')
print('Rows: bikes at location 1 (0..20); columns: bikes at location 2 (0..20).')
print('Positive: location 1 -> location 2; negative: location 2 -> location 1.')
print('      ' + ' '.join(f'{j:>3}' for j in range(21)))
for i in range(21):
    print(f'{i:>3}:  ' + ' '.join(f'{int(basic_policy[i, j]):+3d}' for j in range(21)))

print('\nA few example states (state -> move, expected discounted profit):')
for i, j in [(0, 20), (10, 5), (10, 10), (15, 15), (20, 0)]:
    print(f'({i:2d}, {j:2d}) -> {int(basic_policy[i, j]):+d}, '
          f'value = Rs. {basic_values[i, j]:.3f}')

assert basic_policy.shape == (21, 21)
assert int(basic_policy[20, 0]) == 5


Policy improvement 1: changed states = 318; evaluation sweeps = 213
Policy improvement 2: changed states = 272; evaluation sweeps = 179
Policy improvement 3: changed states = 79; evaluation sweeps = 167
Policy improvement 4: changed states = 8; evaluation sweeps = 132
Policy improvement 5: changed states = 0; evaluation sweeps = 59

Original Gbike: optimal overnight transfers for all 441 states
Rows: bikes at location 1 (0..20); columns: bikes at location 2 (0..20).
Positive: location 1 -> location 2; negative: location 2 -> location 1.
        0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20
  0:   +0  +0  +0  +0  +0  +0  +0  +0  -1  -1  -2  -2  -2  -3  -3  -3  -3  -3  -4  -4  -4
  1:   +0  +0  +0  +0  +0  +0  +0  +0  +0  -1  -1  -1  -2  -2  -2  -2  -2  -3  -3  -3  -3
  2:   +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  -1  -1  -1  -1  -1  -2  -2  -2  -2  -2
  3:   +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  +0  -1  -1  -1  -1  -1  -2
  4:   

## Lab 10 — Question 3: Modified Gbike using policy iteration

**Question:** Solve the Gbike problem using **policy iteration**, but now:

1. The **first bicycle** transferred **from location 1 to location 2** is free each night. Additional 1→2 bicycles cost ₹2 each, and transfers 2→1 always cost ₹2 per bicycle.
2. If a location has **more than 10 bicycles after the overnight transfer**, an additional **₹4** fixed parking charge applies at that location, regardless of how many are over 10.

The state space, Poisson means, 20-bike capacity, five-bike transfer limit, and `γ = 0.9` remain unchanged. The program evaluates a policy, improves it, and repeats until the optimal action no longer changes. The output shows the chosen overnight transfers for all 441 states.

In [9]:
modified_gbike = GbikeMDP(modified=True)
optimal_policy, optimal_values = modified_gbike.policy_iteration()

print('\nOptimal transfer policy (all 441 states):')
print('Rows = bikes at location 1 (0..20); columns = bikes at location 2 (0..20).')
print('Positive: move bikes 1 -> 2; negative: move bikes 2 -> 1.')
print('      ' + ' '.join(f'{j:>3}' for j in range(21)))
for i in range(21):
    print(f'{i:>3}:  ' + ' '.join(f'{int(optimal_policy[i, j]):+3d}'
                              for j in range(21)))

print('\nSelected examples (state -> optimal transfer, expected discounted profit):')
for i, j in [(0, 20), (10, 5), (10, 10), (15, 15), (20, 0)]:
    print(f'({i:2d}, {j:2d}) -> {int(optimal_policy[i, j]):+d}, '
          f'value = Rs. {optimal_values[i, j]:.3f}')

Policy improvement 1: changed states = 371; evaluation sweeps = 213
Policy improvement 2: changed states = 265; evaluation sweeps = 186
Policy improvement 3: changed states = 102; evaluation sweeps = 171
Policy improvement 4: changed states = 0; evaluation sweeps = 139

Optimal transfer policy (all 441 states):
Rows = bikes at location 1 (0..20); columns = bikes at location 2 (0..20).
Positive: move bikes 1 -> 2; negative: move bikes 2 -> 1.
        0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20
  0:   +0  +0  +0  +0  +0  +0  +0  -1  -1  -2  -2  -3  -3  -3  -4  -5  -4  -4  -5  -5  -5
  1:   +1  +0  +0  +0  +0  +0  +0  +0  -1  -1  -2  -2  -2  -3  -4  -5  -3  -4  -4  -4  -4
  2:   +1  +1  +0  +0  +0  +0  +0  +0  +0  -1  -1  -1  -2  -3  -4  -5  -3  -3  -3  -3  -3
  3:   +1  +1  +1  +1  +0  +0  +0  +0  +0  +0  +0  -1  -2  -3  -4  -5  -2  -2  -2  -2  -2
  4:   +1  +1  +1  +1  +1  +0  +0  +0  +0  +0  +0  -1  -2  -3  -4  -1  -1  -1  -1  -1  -1
  5:   +1  +1 

**What we did / result:** Calculated the optimal number of overnight transfers for each possible pair of bike counts with the new free-transfer and parking rules.

**Relation to Question 2:** Both cases use the same rental and return distributions. Their only difference is the revised moving and parking costs, so the selected transfer decisions can change.
